In [1]:
import anndata as ad
import pandas as pd
import numpy as np
import mofax as mfx
import duckdb as db
def limpiar_drug(s):
    return (s.str.strip()
             .str.upper()
             .str.replace(' ', '_', regex=False)  
             .str.replace(r'_+', '_', regex=True)
             .str.strip('_'))

In [20]:
# --- RUTAS ---
MOFA_MODEL = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/modelo_prueba_final_fast_convergence/modelo_mofa_30factors.hdf5'
INPUT_PARQUET = "/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/datos/datos_con_placa_14/tidy_final.parquet"
DRUG_PARQUET = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/drug.parquet'
OUTPUT_H5AD = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/modelo_prueba_final_fast_convergence/adata/mofa_adata_30f.h5ad'

# 1. Cargar modelo MOFA
print("Cargando modelo MOFA...")
model = mfx.mofa_model(MOFA_MODEL)

# 2. Factores (muestras x factores)
Z = model.get_factors(df=True)
print(f"Factores: {Z.shape}")
print(f"Índice ejemplo: {Z.index[:3].tolist()}")

# 3. Metadata desde el índice de los factores
# sample formato: drug_concentracion_plate (plate es el último segmento tras _)
split = Z.index.to_series().str.rsplit('_', n=2, expand=True)
plate = split[2]
concentration = split[1]
drug = split[0]
obs = pd.DataFrame({
    'drug': limpiar_drug(drug).values,
    'concentration': concentration.values,
    'plate' : plate.values
}, index=Z.index)

# 4. Añadir MOA desde drug metadata
print("Añadiendo MOA...")
drug_meta = pd.read_parquet(DRUG_PARQUET)
drug_meta['drug'] = limpiar_drug(drug_meta['drug'])

if 'moa-fine' in drug_meta.columns:
    moa_map = drug_meta[['drug', 'moa-fine','moa-broad']].drop_duplicates('drug')
    obs = obs.merge(moa_map, on='drug', how='left')
    obs.index = Z.index
    print(f"  MOA añadido. NaN: {obs['moa-fine'].isna().sum()}")
else:
    print(f"  Columnas disponibles en drug_meta: {drug_meta.columns.tolist()}")
    print("  Ajusta el nombre de la columna MOA manualmente")

# 5. Pesos de MOFA
print("Extrayendo pesos MOFA...")

# 6. Crear AnnData
print("Creando AnnData...")
adata = ad.AnnData(
    X=Z.values,
    obs=obs,
    var=pd.DataFrame(index=Z.columns),
)

# 7. Guardar pesos en uns — media correcta entre views (líneas celulares)
views = list(model.get_views())
W_per_view = {view: model.get_weights(views=view, df=True) for view in views}

# Guardar pesos individuales por view
for view, w in W_per_view.items():
    adata.uns[f'mofa_weights_{view}'] = w.values
    adata.uns[f'mofa_weights_genes_{view}'] = w.index.tolist()
adata.uns['mofa_weights_factors'] = list(Z.columns)
adata.uns['mofa_views'] = views

# Media entre views alineando genes → matriz consenso para decoupler
genes_ref = list(W_per_view.values())[0].index
W_array = np.stack(
    [w.reindex(genes_ref).values for w in W_per_view.values()], axis=0
)  # (n_views, n_genes, n_factors)
W_mean = np.nanmean(W_array, axis=0)  # ignora NaN en la media

# Después sí puedes eliminar genes que son NaN en TODAS las views
nan_en_todas = np.isnan(W_array).all(axis=0).all(axis=1)  # genes sin datos en ninguna view
print(f"Genes sin datos en ninguna view: {nan_en_todas.sum()}")

W_mean = W_mean[~nan_en_todas]
genes_ref_clean = genes_ref[~nan_en_todas]

adata.uns['mofa_weights'] = W_mean
adata.uns['mofa_weights_genes'] = genes_ref_clean.tolist()

print(f"Pesos consenso: {W_mean.shape} ({len(genes_ref)} genes x {W_mean.shape[1]} factores)")
print(f"Filas con todos ceros: {(W_mean == 0).all(axis=1).sum()}")

# 8. Limpiar columnas object con NaN antes de guardar
for col in adata.obs.columns:
    if adata.obs[col].dtype == object:
        adata.obs[col] = adata.obs[col].fillna('unknown').astype(str)

# 9. Guardar
print(f"\n{adata}")
print(f"\nobs columns: {adata.obs.columns.tolist()}")
adata.write(OUTPUT_H5AD, compression='gzip')
print(f"Guardado en {OUTPUT_H5AD}")

Cargando modelo MOFA...
Factores: (1297, 30)
Índice ejemplo: ['4EGI-1_0.05_1', '9-ING-41_0.05_1', 'APTO-253_0.05_1']
Añadiendo MOA...
  MOA añadido. NaN: 0
Extrayendo pesos MOFA...
Creando AnnData...
Genes sin datos en ninguna view: 0
Pesos consenso: (18775, 30) (18775 genes x 30 factores)
Filas con todos ceros: 14

AnnData object with n_obs × n_vars = 1297 × 30
    obs: 'drug', 'concentration', 'plate', 'moa-fine', 'moa-broad'
    uns: 'mofa_weights_A-172', 'mofa_weights_genes_A-172', 'mofa_weights_A-427', 'mofa_weights_genes_A-427', 'mofa_weights_A498', 'mofa_weights_genes_A498', 'mofa_weights_A549', 'mofa_weights_genes_A549', 'mofa_weights_AN3 CA', 'mofa_weights_genes_AN3 CA', 'mofa_weights_ASPC-1', 'mofa_weights_genes_ASPC-1', 'mofa_weights_BT-474', 'mofa_weights_genes_BT-474', 'mofa_weights_C-33 A', 'mofa_weights_genes_C-33 A', 'mofa_weights_C32', 'mofa_weights_genes_C32', 'mofa_weights_CFPAC-1', 'mofa_weights_genes_CFPAC-1', 'mofa_weights_CHP-212', 'mofa_weights_genes_CHP-212', '

In [21]:
adata.uns['mofa_weights'].shape

(18775, 30)

In [10]:
adata.obs

,drug,concentration,plate,moa-fine,moa-broad
4EGI-1_0.05_1,4EGI-1_0.05,1,1,NaN,NaN
9-ING-41_0.05_1,9-ING-41_0.05,1,1,NaN,NaN
APTO-253_0.05_1,APTO-253_0.05,1,1,NaN,NaN
AT7519_0.05_1,AT7519_0.05,1,1,NaN,NaN
AZD1390_0.05_1,AZD1390_0.05,1,1,NaN,NaN
...,...,...,...,...,...
TRAMETINIB_(DMSO_TF SOLVATE)_5.0_9,TRAMETINIB_(DMSO_TF_SOLVATE)_5.0,9,9,NaN,NaN
TRANILAST_5.0_9,TRANILAST_5.0,9,9,NaN,NaN
VERAPAMIL_5.0_9,VERAPAMIL_5.0,9,9,NaN,NaN
VERTEPORFIN_5.0_9,VERTEPORFIN_5.0,9,9,NaN,NaN


In [7]:
split

,0,1
4EGI-1_0.05_1,4EGI-1_0.05,1
9-ING-41_0.05_1,9-ING-41_0.05,1
APTO-253_0.05_1,APTO-253_0.05,1
AT7519_0.05_1,AT7519_0.05,1
AZD1390_0.05_1,AZD1390_0.05,1
...,...,...
TRAMETINIB_(DMSO_TF SOLVATE)_5.0_9,TRAMETINIB_(DMSO_TF SOLVATE)_5.0,9
TRANILAST_5.0_9,TRANILAST_5.0,9
VERAPAMIL_5.0_9,VERAPAMIL_5.0,9
VERTEPORFIN_5.0_9,VERTEPORFIN_5.0,9
